# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Binary classification.**

Lane 2 (Refresh / Content Opportunity Scoring) asks "which pages should be reviewed first?" — that's fundamentally a ranking problem, but the cleanest way to get there is to first classify each page into `worth reviewing now` (1) vs `not a priority right now` (0), then rank by the model's predicted probability. This matches exactly what the starter pipeline itself does (scripts/03_train_model.py): a binary classifier scored and sorted for Precision@K, not a raw regression on some continuous "urgency" number that doesn't exist in the data.

In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Task type check -- unique trend_direction values (the source of my label):")
print(df["trend_direction"].value_counts())

Task type check -- unique trend_direction values (the source of my label):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

**Target (starter-level proxy):** is_declining_label = (trend_direction == "down"). This is the same label the starter pipeline already uses — I'm inheriting it deliberately so my Week 5 model has an honest, already-validated floor to beat (Precision@50 = 0.240 baseline / 0.740 random forest).

**Why it's a proxy, not the ideal target:** trend_direction is calculated from the current window, not a future outcome — so it tells me a page is declining right now, not that it will keep declining. Per the lane guide, a stronger capstone-grade target predicts a future window: features from prior 90 days -> decline over the next 30 days. I'll upgrade to that once I have the full warehouse's daily fact table from Week 3 onward, since it has real day-by-day history to build a forward-looking label from. For now, the current-window proxy is the right choice for a leakage-safe first model.

In [ ]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Rows: {len(df):,} | declining rate (positive class): {df['is_declining_label'].mean():.1%}")
print("Class balance is reasonable (not a rare-event problem) -- no heavy resampling needed to start.")

Rows: 30,000 | declining rate (positive class): 54.2%
Class balance is reasonable (not a rare-event problem) -- no heavy resampling needed to start.


## 3. Success metric

**Primary metric: Precision@50 (and Precision@K generally).** The decision this supports is "which pages does a reviewer open first," which is inherently a top-of-list problem — a reviewer with limited time cares about whether the top 20-50 picks are right, not about accuracy across all 30,000 pages (most of which nobody will ever look at this week). This is exactly why the starter pipeline reports Precision@50 as its headline number instead of plain accuracy.

**Secondary metrics I'll track starting Week 5:** ROC-AUC and average precision, to make sure the model generalizes and isn't just tuned to the top-50 cutoff by luck, plus Recall on the declining-with-demand subset, since missing a real decline (a false negative) is the more expensive failure mode I named in Task 2.

In [ ]:
import json
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
print(f"Floor to beat (starter baseline rule), Precision@50: {base:.3f}")
print("Best model available already (random forest), Precision@50:", res["models"]["random_forest"]["precision_at_50"])
print("I will report this same metric, on the same client-holdout split, every week from here on.")

Floor to beat (starter baseline rule), Precision@50: 0.240
Best model available already (random forest), Precision@50: 0.74
I will report this same metric, on the same client-holdout split, every week from here on.


## 4. The unit of analysis, as a real dataframe

**One row = one content page, evaluated at one point in time** (in the starter data, the "current" snapshot; in the full warehouse from Week 3, a specific report_date). Below is the actual dataframe slice with the columns that make up a single unit of analysis: identifiers, the observable features, and the label.

In [ ]:
unit_cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
             "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label"]
print("Unit of analysis -- one row per page:")
df[unit_cols].head(5)

Unit of analysis -- one row per page:


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,2803.0,down,1


## 5. Why ML beats a fixed rule here

A fixed rule like "flag pages that are stale AND still visible" (the starter baseline) can only combine a small, hand-picked set of thresholds a person guessed at. I already have direct proof this isn't enough: in Task 1, that exact hand rule scored Precision@50 = 0.240, while a random forest trained on the same features scored 0.740 — about 3x better at picking the right 50 pages first. The reason is that real decline risk depends on interactions between features (e.g. a page's position, CTR, and word count together, not any one threshold alone) that are impractical to hand-tune but easy for a tree-based model to discover from data. A fixed rule also can't adapt as the definition of "worth reviewing" shifts across content types or intents — a model can pick up on those patterns automatically, as long as I keep it leakage-safe and validate it honestly (client-holdout split, no future-window features, no trend_pct/trend_direction leaking into the feature set).

In [ ]:
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Hand rule Precision@50: {base:.3f}  vs  learned model Precision@50: {rf:.3f}  ({rf/base:.1f}x)")
print("This lift is the evidence a fixed rule is leaving real signal on the table.")


Hand rule Precision@50: 0.240  vs  learned model Precision@50: 0.740  (3.1x)
This lift is the evidence a fixed rule is leaving real signal on the table.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.